# Week 1 Day 2 Lab - Introducing UnderCover

## LLMs and Agents on a commercial task: insurance claims

A claims handler at UnderCover Travel Insurance has already decided each of the 25 claims in its queue. We send every claim to a few models, along with the customer's policy record and the policy wording. Each model replies with a decision of approve, decline or refer, a rationale and a payout. Then we compare the answers with the handlers' decisions and look at what each model cost.

At the end we give the same job to a LangChain agent that uses tools to look up the policy and record its decision.

## 4 parts:

PART 1: Examine the data

PART 2: Run with some models

PART 3: Optional: larger models and Jev

PART 4: An Agent via LangChain

<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../../../assets/stop.png" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#EF476F;">Before you start</h2>
            <span style="color:#EF476F;">This lab calls paid models through OpenRouter and costs only a few cents. If you're OK to spend that, you'll need some credit on your account at <a href="https://openrouter.ai">openrouter.ai</a>. If you'd rather not spend anything, there's a free version of this lab in <a href="week1day2free.ipynb">week1day2free.ipynb</a>.<br/><br/>
            Please execute each cell and check that you get similar results to mine (the models vary a little from run to run). Add print statements and try small adjustments to make the code your own.<br/><br/>
            Some cells have code that isn't about AI, like the charts. It's not important to understand those in detail.
            </span>
        </td>
    </tr>
</table>

<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../../../assets/resources.png" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#118AB2;">Resources</h2>
            <span style="color:#118AB2;">The <a href="../../../guides/01_intro.ipynb">guides</a> folder in this repo has more information on topics like the command line, Git, Python and debugging.<br/><br/>
            If this lab is too much, consider courses 1, 2, 4 and 5 in <a href="https://edwarddonner.com/curriculum">my curriculum</a>: AI Builder, AI Coder, and the Core and Agentic Tracks of AI Engineer.<br/><br/>
            If it's too slow, put me on 2x and zip through it! There's some interesting content on Jev for the pros.
            </span>
        </td>
    </tr>
</table>

<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../../../assets/connect.png" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#118AB2;">Stay in touch</h2>
            <span style="color:#118AB2;">Please message me in Udemy anytime with questions, or email me at <a href="mailto:ed@edwarddonner.com">ed@edwarddonner.com</a>.<br/><br/>
            Let's connect on <a href="https://www.linkedin.com/in/eddonner/">LinkedIn</a>! And subscribe to my <a href="https://www.youtube.com/@edward.donner">YouTube channel</a> for more on LLMs, agents and coding agents.
            </span>
        </td>
    </tr>
</table>

In [ ]:
import json
import os
import time
from pathlib import Path
from typing import Literal

import pandas as pd
import plotly.express as px
from dotenv import load_dotenv
from IPython.display import Markdown, display
from langchain.agents import create_agent
from langchain_core.tools import tool
from langchain_openai import ChatOpenAI
from openai import OpenAI
from openrouter import OpenRouter
from pydantic import BaseModel, Field
import time

## Connect to OpenRouter - and please read note here

The OpenRouter key comes from the `.env` file you made on day 1 in the `undercover` folder.

### NOTE:

If this is a new OpenRouter account, then you might get an error at some point from OpenRouter like: "new accounts are limited to 20 requests per minute for this model"

If you get that error, then add this line immediately after calling an LLM:

`time.sleep(3)`

and you should be in good shape.

In [ ]:
load_dotenv(override=True)
openrouter_api_key = os.getenv("OPENROUTER_API_KEY")
OPENROUTER_BASE_URL = "https://openrouter.ai/api/v1"

if openrouter_api_key:
    print("OpenRouter API key found")
else:
    print("No OpenRouter API key found. Add OPENROUTER_API_KEY to the .env file")

openrouter = OpenAI(base_url=OPENROUTER_BASE_URL, api_key=openrouter_api_key, timeout=120, max_retries=5)

<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../../../assets/info.png" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#118AB2;">Reminder: the Chat Completions API</h2>
            <span style="color:#118AB2;">We call models with the Chat Completions API. <code>messages</code> is a Python list of dicts, each with a <code>role</code> and <code>content</code>. The system prompt tells the model how to behave and the user prompt has the request. The reply is in <code>response.choices[0].message.content</code>.<br/><br/>
            Here's a quick test.
            </span>
        </td>
    </tr>
</table>

In [ ]:
messages = [
    {"role": "system", "content": "You are a helpful assistant who answers in the style of a rabbi."},
    {"role": "user", "content": "What is the meaning of the universe?"}
]

response = openrouter.chat.completions.create(model="openai/gpt-6-luna", messages=messages)
display(Markdown(response.choices[0].message.content))

## Part 1: Load and examine the data

The seed data is in the repo's `data` folder. `get_sample_data` returns the claims and the handlers' decisions on them, keyed by claim id. The model is sent the claims, the policies and the policy wording.

By default we work with the queue of 25 claims. If you are using free models with a daily limit, set `TINY = True` to keep 5 of them.

In [ ]:
DATA = Path("../../../data")


def load(name):
    """Read one of the JSON seed files."""
    return json.loads((DATA / name).read_text(encoding="utf-8"))


def get_sample_data(tiny=False, held_out=False):
    """Return a list of claims, and the handlers' decisions on them keyed by claim id.

    held_out=True gives the twenty-five held-out claims in place of the queue.
    tiny=True keeps the five claims marked for the tiny version of either set.
    """
    claims = load("evals/held_out_claims.json" if held_out else "claims.json")
    decisions = load("evals/held_out_decisions.json" if held_out else "evals/decisions.json")
    if tiny:
        chosen = {decision["claim_id"] for decision in decisions if decision["tiny"]}
        claims = [claim for claim in claims if claim["claim_id"] in chosen]
        decisions = [decision for decision in decisions if decision["tiny"]]
    labels = {decision["claim_id"]: decision for decision in decisions}
    return claims, labels

In [ ]:
TINY = False

claims, labels = get_sample_data(tiny=TINY)
policies = {policy["policy_id"]: policy for policy in load("policies.json")}
policy_wording = (DATA / "policy_wording.md").read_text(encoding="utf-8")

print(f"{len(claims)} claims, {len(labels)} labels, {len(policies)} policies")
print(f"The policy wording is {len(policy_wording.split()):,} words")

In [ ]:
claims[0]

Here is the handler's decision on that claim which we'll use to validate the LLM results.

In [ ]:
labels[claims[0]["claim_id"]]

### Explore the data

It's always good to dig into the data before running experiments.

In [ ]:
rows = []
for claim in claims:
    label = labels[claim["claim_id"]]
    rows.append({
        "claim_id": claim["claim_id"],
        "summary": claim["summary"],
        "claim_type": claim["claim_type"],
        "amount_claimed": claim["amount_claimed"],
        "difficulty": label["difficulty"],
        "decision": label["decision"],
        "payout": label["payout"],
    })
#CLAUDE: This table
data = pd.DataFrame(rows)
data

Claim CLM-1015 might be a familiar example...

In [ ]:
home_alone = [claim for claim in claims if claim["claim_id"] == "CLM-1015"][0]
decision = labels["CLM-1015"]

display(Markdown(f"""**{home_alone['summary']}**

{home_alone['story']}

**Handler:** {decision['decision']} ${decision['payout']:,.2f}. {decision['reason']}
"""))

Here's another one that might ring a bell...

In [ ]:
# Load the whole queue, in case TINY is True
queue, queue_labels = get_sample_data()
hangover = [claim for claim in queue if claim["claim_id"] == "CLM-1010"][0]
decision = queue_labels["CLM-1010"]

display(Markdown(f"""**{hangover['summary']}**

{hangover['story']}

**Handler:** {decision['decision']} ${decision['payout']:,.2f}. {decision['reason']}
"""))

This chart counts the claims of each type, split by the handler's decision.

In [ ]:
fig = px.histogram(data, x="claim_type", color="decision", title="Claims by type")
fig.show()

Here's a breakdown by difficulty. The policy wording is enough to decide the routine and hard claims. The manual claims depend on the claims manual or the customer's history.

In [ ]:
fig = px.histogram(
    data,
    x="difficulty",
    color="decision",
    category_orders={"difficulty": ["routine", "hard", "manual"]},
    title="Claims by difficulty",
)
fig.show()

## Part 2: running with some LLMs

<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../../../assets/info.png" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#118AB2;">Reminder: Structured Outputs</h2>
            <span style="color:#118AB2;">With Structured Outputs the model replies in JSON that follows a schema we give it. We describe the schema as a Pydantic class and pass it to <code>parse()</code> as the <code>response_format</code>. The reply comes back as an object of that class in <code>response.choices[0].message.parsed</code>.
            </span>
        </td>
    </tr>
</table>

Starting by defining a Pydantic class to describe the output so we can use Structured Outputs.

The rationale comes before the decision to ensure the model thinks through the decision before making it.

In [ ]:
class ClaimDecision(BaseModel):
    """A claims handler's decision on one claim."""

    rationale: str = Field(description="2-3 sentences to explain the decision along with the relevant section of the policy")
    decision: Literal["approve", "decline", "refer"]
    payout: float = Field(description="The amount to pay in US dollars, or 0 for a decline or a referral")

## The prompt

The system prompt has the instructions and the whole policy wording. The user prompt has the customer's policy record and the claim, both as JSON.

In [ ]:
INSTRUCTIONS = """You are a claims handler at UnderCover Travel Insurance.
Decide the claim you are given, using the policy wording below and the customer's policy record.
Review the type of the claim.  If it is a personal loss claim then list all the items lost and treat each as per-items lost found in section 5
A pair or set is things that are only used together, such as two shoes, a suit's jacket and trousers, or a camera body and its lens. Several things that can each be used alone are separate items, even when they are bought together or listed on one line, such as three shirts or two swimsuits.

Choose one decision:
- approve: the wording covers the claim and the required documents are attached. Work out the payout using section 10.
- decline: the wording does not cover the claim.
- refer: a colleague needs to look at it, because a required document is missing, or the story and the documents disagree, or the wording does not settle the question.
"""

SYSTEM_PROMPT = f"""{INSTRUCTIONS}
{policy_wording}
"""


def make_user_prompt(claim):
    """The customer's policy record and the claim, as JSON."""
    policy = policies[claim["policy_id"]]
    return f"""# Policy record

{json.dumps(policy, indent=2)}

# Claim

{json.dumps(claim, indent=2)}
"""


def make_messages(claim):
    """Build the messages for one claim: the task and the wording, then the policy record and the claim."""
    return [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": make_user_prompt(claim)},
    ]

### Calling the LLM

In [ ]:
def decide(claim, model, effort):
    """Ask one model to decide one claim."""
    start = time.time()
    response = openrouter.chat.completions.parse(
        model=model,
        messages=make_messages(claim),
        response_format=ClaimDecision,
        reasoning_effort=effort,
        extra_body={"provider": {"require_parameters": True}},
    )
    answer = response.choices[0].message.parsed
    return {
        "model": model,
        "effort": effort,
        "technique": "LLM",
        "claim_id": claim["claim_id"],
        "decision": answer.decision,
        "payout": answer.payout,
        "rationale": answer.rationale,
        "cost": response.usage.cost,
        "seconds": round(time.time() - start, 1),
    }

In [ ]:
first = decide(claims[0], "openai/gpt-6-luna", "low")
first

### Now running multiple models

In [ ]:
def is_right(answer, label):
    """True when the decision matches the handler's and the payout is within a dollar."""
    return answer["decision"] == label["decision"] and abs(answer["payout"] - label["payout"]) <= 1


def run(model, effort, claims, labels):
    """Run the model on the claims and compare against the labels."""
    results = []
    for claim in claims:
        answer = decide(claim, model, effort)
        label = labels[claim["claim_id"]]
        results.append({**answer, "difficulty": label["difficulty"], "right": is_right(answer, label)})
    right = sum(result["right"] for result in results)
    cost = sum(result["cost"] for result in results)
    seconds = sum(result["seconds"] for result in results)
    print(f"{model} at {effort}: {right} of {len(results)} right for ${cost:.4f} in {seconds:.0f} seconds")
    return results

In [ ]:
luna_low = run("openai/gpt-6-luna", "low", claims, labels)

In [ ]:
luna_medium = run("openai/gpt-6-luna", "medium", claims, labels)

### Model comparison

Runs on the queue, scored against the handlers' decisions:

- GPT-6 Luna at `low` and `medium`
- gpt-oss-120b at `low`, `medium` and `high`
- Gemini 3.5 Flash Lite at `low`
- every local model in Ollama, at `low` and `medium` where it supports them (an addition to the lab)

In [ ]:
# Add your own experiments here!
gptoss_low = run("openai/gpt-oss-120b", "low", claims, labels)

In [ ]:
gptoss_med = run("openai/gpt-oss-120b", "medium", claims, labels)

In [ ]:
gptoss_high = run("openai/gpt-oss-120b", "high", claims, labels)

In [ ]:
gemini = run("google/gemini-3.5-flash-lite", "low", claims, labels)

<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../../../assets/exercise.png" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#8338EC;">Addition to the lab: Local models with Ollama</h2>
            <span style="color:#8338EC;">This section is not part of Ed Donner's Week 1 Day 2 lab. We added it to run the models installed on this machine through Ollama, scored with the lab's own <code>is_right</code> and shown in the same tables as the OpenRouter runs. The original lab continues with "Results by difficulty".
            </span>
        </td>
    </tr>
</table>

### Addition to the lab: Local models with Ollama

Every chat model pulled into Ollama on this machine, at low and medium effort where the model supports them. The local models have their own `decide_local` and `run_local`, but get the same claims, prompts and schema and are scored with the same `is_right`, so their results sit in the same tables as the OpenRouter runs. As in the agent section, a claim with no decision counts as wrong. Each model is unloaded after its runs, so only one is in memory at a time. Cost is $0, and the first claim's time includes loading the model.

In [ ]:
import ollama
import requests

OLLAMA_URL = "http://localhost:11434"
EFFORTS = ["low", "medium"]

ollama_client = OpenAI(base_url=f"{OLLAMA_URL}/v1", api_key="ollama", timeout=600)
local_results = {}


def decide_local(claim, model, effort):
    """Ask one local Ollama model to decide one claim, with the same prompt and schema as decide()."""
    start = time.time()
    response = ollama_client.chat.completions.parse(
        model=model,
        messages=make_messages(claim),
        response_format=ClaimDecision,
        **({"reasoning_effort": effort} if effort else {}),
    )
    answer = response.choices[0].message.parsed
    return {
        "decision": answer.decision,
        "payout": answer.payout,
        "rationale": answer.rationale,
        "seconds": round(time.time() - start, 1),
    }


def run_local(model, effort):
    """Run a local model on the claims and score each answer with is_right. A claim that fails counts as wrong."""
    results = []
    for claim in claims:
        start = time.time()
        try:
            answer = decide_local(claim, model, effort)
        except Exception as error:
            print(f"  {claim['claim_id']} failed: {error}")
            answer = {"decision": "no decision", "payout": 0.0, "rationale": f"Failed: {error}", "seconds": round(time.time() - start, 1)}
        label = labels[claim["claim_id"]]
        results.append({
            "model": f"ollama/{model}",
            "effort": effort,
            "technique": "LLM",
            "claim_id": claim["claim_id"],
            **answer,
            "cost": 0,
            "difficulty": label["difficulty"],
            "right": is_right(answer, label),
        })
    right = sum(result["right"] for result in results)
    seconds = sum(result["seconds"] for result in results)
    print(f"ollama/{model} at {effort}: {right} of {len(results)} right in {seconds:.0f} seconds")
    return results

In [ ]:
def local_models():
    """Every model in Ollama on this machine, and which of EFFORTS it supports. None means it runs once with no effort."""
    models = []
    for entry in ollama.list().models:
        info = requests.post(f"{OLLAMA_URL}/api/show", json={"model": entry.model}, timeout=30).json()
        capabilities = info.get("capabilities", [])
        levels = info.get("thinking", {}).get("values", [])
        context = next((v for k, v in info.get("model_info", {}).items() if k.endswith(".context_length")), None)
        models.append({
            "model": entry.model,
            "size_gb": round(entry.size / 1e9, 1),
            "max_context": context,
            "thinking_levels": levels,
            "efforts": [effort for effort in EFFORTS if effort in levels] or [None],
            "usable": "completion" in capabilities,
            "tools": "tools" in capabilities,
            "reason": "" if "completion" in capabilities else "can't chat: no completion capability",
        })
    return models


available = local_models()
pd.DataFrame(available)

In [ ]:
for entry in available:
    if not entry["usable"]:
        continue
    name = entry["model"]
    try:
        for effort in entry["efforts"]:
            if (name, effort) not in local_results:
                local_results[(name, effort)] = run_local(name, effort)
    finally:
        ollama.generate(model=name, keep_alive=0)

## Results by difficulty

This counts the claims each entry got right at each level of difficulty. Expect nearly all of the manual claims to be wrong for now.

In [ ]:
scored = pd.DataFrame(luna_low + luna_medium + gemini + sum(local_results.values(), []))  # local runs added to the lab's table
scored["effort"] = scored["effort"].fillna("none")
scored.pivot_table(index=["model", "effort"], columns="difficulty", values="right", aggfunc="sum", sort=False)

## Where a model got it wrong

For each claim a model got wrong, this shows its rationale, the handler's reason and what the claim is there to test. Here we look at Luna at `low`. Pass in another set of results to see its misses.

In [ ]:
def show_misses(results, labels):
    """Show each claim in a set of results that the model got wrong, with its rationale and the handler's reason."""
    for result in results:
        if not result["right"]:
            label = labels[result["claim_id"]]
            display(Markdown(f"""**{result['claim_id']}, {label['difficulty']}**: {result['decision']} ${result['payout']:,.2f}, where the handler said {label['decision']} ${label['payout']:,.2f}

- Model: {result['rationale']}
- Handler: {label['reason']}
- Tests: {label['tests']}
"""))


show_misses(luna_low, labels)

In [ ]:
for (name, effort), results in local_results.items():
    display(Markdown(f"### {name} at {effort}"))
    show_misses(results, labels)

### What it cost

This shows how many claims each entry got right, what it cost and how long it took for the whole queue.

In [ ]:
scored.groupby(["model", "effort"], sort=False)[["right", "cost", "seconds"]].sum()

<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../../../assets/business.png" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#118AB2;">Commercial implications</h2>
            <span style="color:#118AB2;">Accuracy, cost and speed are the numbers a customer will ask about. Cost per claim is the easiest one to relate to the business: compare it with what a handler's time on a claim costs.<br/><br/>
            The kind of mistake matters too. A wrong approval pays out money the policy doesn't owe, a wrong decline lets a customer down and a wrong referral costs a handler's time.
            </span>
        </td>
    </tr>
</table>

## Part 3: Optional extras

These two experiments on the queue are optional. Nothing later in the lab depends on them.

### Sol at high

GPT-6 Sol is Luna's bigger sibling; we run it at the `high` reasoning level to see how much better it does on this prompt.

In [ ]:
sol = run("openai/gpt-6-sol", "high", claims, labels)

### Jev

Jev, from TypeSafe AI, doesn't generate text. We send it the facts as `state` along with a typed question, and it returns a probability for each option and a confidence. It chooses between approve, decline and refer in a fraction of a second. It can't write a rationale or work out a payout, so we score it on the decision alone.

Jev is on OpenRouter's Decisions endpoint, which the OpenAI client can't reach. We use OpenRouter's own Python client with the same key.

In [ ]:
JEV = "typesafe/jev-1.13"
JEV_QUESTION = {
    "type": "choice",
    "instructions": "You are a claims handler at UnderCover Travel Insurance. Decide `claim` using `policy_wording` and the customer's `policy` record.",
    "criteria": {
        "approve": "The wording covers the claim and the required documents are attached.",
        "decline": "The wording does not cover the claim.",
        "refer": "A colleague needs to look at it, because a required document is missing, or the story and the documents disagree, or the wording does not settle the question.",
    },
}

openrouter_decisions = OpenRouter(api_key=openrouter_api_key, timeout_ms=60_000)


def decide_with_jev(claim):
    """Ask Jev for the decision on one claim, with its confidence."""
    state = {"policy_wording": policy_wording, "policy": policies[claim["policy_id"]], "claim": claim}
    start = time.time()
    response = openrouter_decisions.alpha.decisions.create(model=JEV, state=state, questions={"decision": JEV_QUESTION})
    answer = response.answers["decision"]
    return {
        "claim_id": claim["claim_id"],
        "decision": answer.choice,
        "confidence": answer.confidence,
        "right": answer.choice == labels[claim["claim_id"]]["decision"],
        "cost": response.usage.cost,
        "seconds": round(time.time() - start, 2),
    }

In [ ]:
jev = pd.DataFrame([decide_with_jev(claim) for claim in claims])
luna_decisions_right = sum(result["decision"] == labels[result["claim_id"]]["decision"] for result in luna_low)

print(f"Jev: {jev['right'].sum()} of {len(jev)} decisions right for ${jev['cost'].sum():.4f} in {jev['seconds'].sum():.0f} seconds")
print(f"Luna at low: {luna_decisions_right} of {len(luna_low)} decisions right")

Here are Jev's decisions split at a confidence of 0.9. One way to use a fast model like this is to keep the answers it's confident about and send the rest to a stronger model.

In [ ]:
jev["confident"] = jev["confidence"] >= 0.9
jev.groupby("confident")["right"].agg(["sum", "count"])

<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../../../assets/exercise.png" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#8338EC;">Addition to the lab: Consensus</h2>
            <span style="color:#8338EC;">This section is not part of Ed Donner's Week 1 Day 2 lab. We added it, with <code>consensus.py</code>, to look across every run at the claims they got wrong and ask whether they went wrong for the same reason. The original lab continues with Part 4.
            </span>
        </td>
    </tr>
</table>

# Addition to the lab: what the misses have in common

The comparison tables above score each run on its own. Here we look across every run at once, at the claims they got wrong, and ask whether the runs went wrong in the same way. A claim that many runs miss for the same reason points at the prompt or the information we give the models, not at one model.

The runs compared are the OpenRouter runs from Part 2 and every local Ollama run in `local_results`, so run the Ollama section first. If you skipped it, run its setup cell, which creates an empty `local_results`, and the section compares the OpenRouter runs alone.

### How the code runs

| Step | Cell | Function in `consensus.py` | What it does | Model call |
|---|---|---|---|---|
| 1 | Pool the misses | `pool_misses(experiments)` | Puts every run's answers in one DataFrame, labels each with its run (model, effort and technique) and keeps the wrong ones. A local claim that failed (`"no decision"`) is dropped here: it still counts as wrong in the score tables, but a failed call has no reason to compare | None |
| 2 | Count them | `count_misses(misses)` | Each claim and how many runs missed it | None |
| 3 | Name the runs | `misses_by_run(misses)` | The same count, with the runs that missed each claim | None |
| 4 | Look for agreement | `find_consensus(openrouter, misses, claims, labels)` | For each claim missed by two or more runs, finds the most common wrong outcome in code, then asks the judge whether those answers share a reason. It reports the consensus, how many agreed and how the reason departs from the handler's | **Yes**: the judge, Claude Sonnet on OpenRouter, once per claim with a majority outcome. The cost is printed |
| 5 | Show the detail | `show_consensus(consensus, labels)` | For each claim where the misses agree: the shared reason, the gap from the handler and one example answer | None |

### Things to know

- **Consensus is not truth.** The handler's decision is the truth, and every answer here is one the handler disagreed with.
- **More runs, more judging.** With the local runs included, more claims are missed by two or more runs, so step 4 makes more paid calls.
- **Labels.** Local runs show as, for example, `ollama/qwen3.8:27b at low (LLM)`. Models run with no effort show `at None`.

With Claude's help

## Which claims get missed, across every run

`show_misses` looks at one run at a time. Here we pool the results of every run into one DataFrame, so each row is one run's answer on one claim, and keep the rows the model got wrong.

Then we group the misses by claim id to make two tables:

1. Each claim and the number of runs that missed it
2. The same count, with the runs that missed it

Luna at `low` and Luna at `medium` share a model name, so each run gets a label made from its model, reasoning effort and technique. A claim that every run got right doesn't appear in either table. A claim missed by every run is one this prompt can't settle, whichever model reads it.

The code is in `consensus.py`, next to this notebook, so later labs can import it and run the same comparison.

With Claude's help

In [ ]:
from consensus import count_misses, find_consensus, misses_by_run, pool_misses, show_consensus

# Add any other runs you have made to this list. The local Ollama runs are included.
experiments = [luna_low, luna_medium, gemini, *local_results.values()]

misses = pool_misses(experiments)
misses = misses[misses["decision"] != "no decision"]  # a failed call isn't a reason, so keep it away from the judge
print(f"{len(misses)} misses across {len(experiments)} runs of {len(claims)} claims")

In [ ]:
count_misses(misses)

The same table again, this time naming the runs that missed each claim.

In [ ]:
misses_by_run(misses)

## Do the misses agree with each other?

When several runs miss the same claim, it helps to know whether they went wrong in the same way. If they did, the prompt or the information we gave them is leading every model to the same place. If their reasons scatter, the claim is ambiguous or the models are noisy on it.

This is a measure of consensus, not of truth. The handler's decision is still the truth here, and every answer below is one the handler disagreed with. The runs all read the same prompt, so they can easily share a blind spot.

For each claim missed by two or more runs:

1. Group the missed answers by decision and payout, in code. Answers with different outcomes don't agree, whatever their rationales say.
2. If one outcome has more than half of the misses, a judge model sorts those answers by their reason: the same policy section and the same deciding fact.
3. If one reason has more than half of the misses, that's the consensus, and we show one of its answers as the example. Otherwise there is no agreement.

The judge also says how the consensus departs from the handler's reason. The judge, set by `JUDGE` in `consensus.py`, is Claude Sonnet 5.5, from a different family than the models it's judging. It sorts the answers and the code does the counting.

With Claude's help

In [ ]:
consensus = find_consensus(openrouter, misses, claims, labels)

print(f"{len(consensus)} claims missed by two or more runs, judged for ${consensus['cost'].sum():.4f}")
consensus[["claim_id", "times_missed", "consensus", "agreed", "gap"]]

Here is the detail for each claim where the misses agree: the shared reason, where it departs from the handler's, and one of the answers as the example.

In [ ]:
show_consensus(consensus, labels)

## Part 4: Using LangChain create_agent

Our approaches so far have used a single LLM call with Structured Outputs.

Now we'll use a different approach: an AI agent.

An LLM, with tools, in a loop, to achieve a goal!

We'll use LangChain's `create_agent` to run the loop for us.

We give it 2 tools. `get_policy` looks up the customer's policy record. `record_decision` saves the decision.

<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../../../assets/info.png" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#118AB2;">Reminder: LangChain create_agent</h2>
            <span style="color:#118AB2;">1. Describe the LLM with a LangChain class like <code>ChatOpenAI</code><br/>
            2. Turn a function into a tool with <code>@tool</code><br/>
            3. <code>create_agent</code> builds the agent and <code>invoke()</code> runs its loop<br/><br/>
            Behind the scenes, a LangGraph graph is created; we'll find out more in Week 2.
            </span>
        </td>
    </tr>
</table>

In [ ]:
llm = ChatOpenAI(model="openrouter/free", base_url=OPENROUTER_BASE_URL, api_key=openrouter_api_key)

messages = [
    {"role": "system", "content": "You are a helpful assistant who answers in the style of a pirate."},
    {"role": "user", "content": "What is the meaning of the universe?"},
]

response = llm.invoke(messages)
display(Markdown(response.content))

In [ ]:
recorded = {}


@tool
def get_policy(policy_id: str) -> str:
    """Look up a customer's policy record: the plan, the add-ons, the purchase date, the trip and the travelers."""
    policy = policies.get(policy_id)
    return json.dumps(policy, indent=2) if policy else f"There is no policy {policy_id}"


@tool
def record_decision(claim_id: str, decision: Literal["approve", "decline", "refer"], payout: float, rationale: str) -> str:
    """Record the decision on a claim in the claims system.

    payout is the amount to pay in US dollars, or 0 for a decline or a referral.
    rationale is 2-3 sentences to explain the decision along with the relevant section of the policy.
    """
    recorded[claim_id] = {"decision": decision, "payout": payout, "rationale": rationale}
    return f"Recorded {decision} on {claim_id}"

The agent uses Luna at `low` through LangChain's `ChatOpenAI`, pointed at OpenRouter. Its system prompt has the same instructions and policy wording as before, plus a line about the tools. Once the agent has looked up the policy it has the same information as before, so we can compare its score with Luna at `low`.

In [ ]:
luna = ChatOpenAI(model="openai/gpt-6-luna", base_url=OPENROUTER_BASE_URL, api_key=openrouter_api_key, reasoning_effort="low", timeout=120, max_retries=5)

AGENT_SYSTEM_PROMPT = f"""{INSTRUCTIONS}
Use your tools to look up the customer's policy record and to record your decision.

{policy_wording}
"""


def make_claim_prompt(claim):
    """The claim alone, as JSON. The agent looks up the policy record for itself."""
    return f"""# Claim

{json.dumps(claim, indent=2)}
"""


agent = create_agent(model=luna, tools=[get_policy, record_decision], system_prompt=AGENT_SYSTEM_PROMPT)

First, trying out one claim. The model asks for the policy, the tool returns it, the model records its decision and then finishes with a short message.

In [ ]:
result = agent.invoke({"messages": [{"role": "user", "content": make_claim_prompt(claims[0])}]})

for message in result["messages"][1:]:
    message.pretty_print()

Now the whole queue. Each claim takes three model calls: one to ask for the policy, one to record the decision and one to finish. OpenRouter's cost for each call is in the message metadata. A claim with no recorded decision counts as wrong.

In [ ]:
def decide_with_agent(claim):
    """Run the agent on one claim and check what it recorded against the handler's decision."""
    start = time.time()
    result = agent.invoke({"messages": [{"role": "user", "content": make_claim_prompt(claim)}]})
    answer = recorded.get(claim["claim_id"], {"decision": "nothing recorded", "payout": 0.0, "rationale": ""})
    label = labels[claim["claim_id"]]
    cost = sum(message.response_metadata["token_usage"]["cost"] for message in result["messages"] if message.type == "ai")
    return {
        "model": "openai/gpt-6-luna",
        "effort": "low",
        "technique": "Agent",
        "claim_id": claim["claim_id"],
        **answer,
        "difficulty": label["difficulty"],
        "right": is_right(answer, label),
        "cost": cost,
        "seconds": round(time.time() - start, 1),
    }


recorded.clear()
agent_results = []
for claim in claims:
    agent_results.append(decide_with_agent(claim))

print(f"{len(recorded)} of {len(claims)} claims have a recorded decision")

In [ ]:
pd.DataFrame(luna_low + agent_results).groupby(["model", "effort", "technique"], sort=False)[["right", "cost", "seconds"]].sum()

In [ ]:
show_misses(agent_results, labels)

<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../../../assets/exercise.png" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#8338EC;">Addition to the lab: The agent on every model</h2>
            <span style="color:#8338EC;">This section is not part of Ed Donner's Week 1 Day 2 lab. We run Part 4's agent, with the same tools, prompt and scoring, on the models the single-call runs used: the OpenRouter models, with Sol at high from Part 3, and the local Ollama models. A local model has to be able to call tools to drive the agent, so models without that are skipped. Each model runs twice, and passes when both runs score at least 15 of 25. The original lab continues with "Putting it all together".
            </span>
        </td>
    </tr>
</table>

In [ ]:
GOAL = 15  # of 25; use 3 when TINY is True
RUNS = 2

AGENT_RUNS = [  # every OpenRouter model and effort from Parts 2 and 3
    ("openai/gpt-6-luna", "low"),
    ("openai/gpt-6-luna", "medium"),
    ("openai/gpt-oss-120b", "low"),
    ("openai/gpt-oss-120b", "medium"),
    ("openai/gpt-oss-120b", "high"),
    ("google/gemini-3.5-flash-lite", "low"),
    ("openai/gpt-6-sol", "high"),
]


def agent_llm(model, effort):
    """A LangChain chat model: a local Ollama model when the name starts with ollama/, otherwise OpenRouter."""
    options = {"reasoning_effort": effort} if effort else {}
    if model.startswith("ollama/"):
        return ChatOpenAI(model=model.removeprefix("ollama/"), base_url=f"{OLLAMA_URL}/v1", api_key="ollama", timeout=600, **options)
    return ChatOpenAI(model=model, base_url=OPENROUTER_BASE_URL, api_key=openrouter_api_key, timeout=120, max_retries=5, **options)


def run_agent(model, effort):
    """Part 4's agent, tools and prompt on another model, scored like decide_with_agent. A claim with no recorded decision counts as wrong."""
    claim_agent = create_agent(model=agent_llm(model, effort), tools=[get_policy, record_decision], system_prompt=AGENT_SYSTEM_PROMPT)
    results = []
    for claim in claims:
        recorded.pop(claim["claim_id"], None)
        start = time.time()
        cost = 0.0
        try:
            result = claim_agent.invoke({"messages": [{"role": "user", "content": make_claim_prompt(claim)}]})
            cost = sum((message.response_metadata.get("token_usage") or {}).get("cost") or 0 for message in result["messages"] if message.type == "ai")
        except Exception as error:
            print(f"  {claim['claim_id']} failed: {error}")
        answer = recorded.get(claim["claim_id"], {"decision": "nothing recorded", "payout": 0.0, "rationale": ""})
        label = labels[claim["claim_id"]]
        results.append({
            "model": model,
            "effort": effort,
            "technique": "Agent",
            "claim_id": claim["claim_id"],
            **answer,
            "difficulty": label["difficulty"],
            "right": is_right(answer, label),
            "cost": cost,
            "seconds": round(time.time() - start, 1),
        })
    right = sum(result["right"] for result in results)
    total = sum(result["cost"] for result in results)
    seconds = sum(result["seconds"] for result in results)
    print(f"{model} at {effort} (Agent): {right} of {len(results)} right for ${total:.4f} in {seconds:.0f} seconds")
    return results

Every model through the agent, twice: the OpenRouter models in `AGENT_RUNS`, then each local model that can call tools, at each effort it supports. The OpenRouter runs are paid, at about three calls a claim. Each local model is unloaded after its runs.

In [ ]:
agent_models = AGENT_RUNS + [(f"ollama/{entry['model']}", effort) for entry in available if entry["usable"] and entry["tools"] for effort in entry["efforts"]]

agent_rows = []
for model, effort in agent_models:
    for run in range(1, RUNS + 1):
        agent_rows += [{**result, "run": run} for result in run_agent(model, effort)]
    if model.startswith("ollama/"):
        ollama.generate(model=model.removeprefix("ollama/"), keep_alive=0)

In [ ]:
agent_table = pd.DataFrame(agent_rows)
agent_table["effort"] = agent_table["effort"].fillna("none")

verdict = agent_table.pivot_table(index=["model", "effort"], columns="run", values="right", aggfunc="sum", sort=False)
verdict.columns = [f"run {run}" for run in verdict.columns]
verdict["passes"] = (verdict >= GOAL).all(axis=1)
verdict

## Putting it all together

Here is every experiment in one table. Cost is for the whole queue and speed is the average time for one claim. Jev isn't included because it only makes the decision.

In [ ]:
# If you skipped Part 3, take sol out of this list
experiments = [luna_low, luna_medium, gemini, sol, agent_results]

rows = []
for results in experiments:
    right = sum(result["right"] for result in results)
    cost = sum(result["cost"] for result in results)
    seconds = sum(result["seconds"] for result in results)
    rows.append({
        "LLM": results[0]["model"],
        "reasoning": results[0]["effort"],
        "technique": results[0]["technique"],
        "accuracy": f"{right} / {len(results)}",
        "cost": f"${cost:.4f}",
        "speed": f"{seconds / len(results):.1f} seconds a claim",
    })

pd.DataFrame(rows)

### Addition to the lab: every run, including the additions

Donner's table above, extended to every single-call run and every agent run, on OpenRouter and on Ollama. Each agent run has its own row, numbered by `run`. Sol is in both, and Jev in neither, as in the lab.

In [ ]:
llm_runs = [luna_low, luna_medium, gptoss_low, gptoss_med, gptoss_high, gemini, sol, *local_results.values()]  # take out any you skipped
agent_run_list = [group.to_dict("records") for _, group in agent_table.groupby(["model", "effort", "run"], sort=False)]

rows = []
for results in llm_runs + agent_run_list:
    rows.append({
        "LLM": results[0]["model"],
        "reasoning": results[0]["effort"] or "none",
        "technique": results[0]["technique"],
        "run": results[0].get("run", 1),
        "right": sum(result["right"] for result in results),
        "of": len(results),
        "cost": round(sum(result["cost"] for result in results), 4),
        "seconds a claim": round(sum(result["seconds"] for result in results) / len(results), 1),
    })

summary = pd.DataFrame(rows).sort_values(["technique", "right"], ascending=[True, False])
summary.to_csv(Path.home() / "week1day2_results.csv", index=False)
summary

<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../../../assets/exercise.png" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#EF476F;">Your assignment</h2>
            <span style="color:#EF476F;">This lab is about experimenting with different LLMs and Agents to understand the accuracy, cost and speed.<br/><br/>
            Based on our initial experiments, Luna (low) is able to perform well at a low cost. More expensive models can perform better.<br/><br/>
            Challenge: with a goal of at least 15 / 25 correct consistently, using the agent, what is the cheapest model you can find?<br/><br/>
            I'd love it if you would consider making a PR with your version of the lab in the <code>community_contributions</code> folder. There are instructions on making a PR at <a href="https://edwarddonner.com/pr">edwarddonner.com/pr</a>.
            </span>
        </td>
    </tr>
</table>

<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../../../assets/thanks.png" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#069D75;">Congratulations! And if I could disturb you with a quick request:</h2>
            <span style="color:#069D75;">You've finished the first main lab. You compared several models and an agent on UnderCover's claims, with accuracy, cost and speed side by side.<br/><br/>
            On Day 3 we bring in coding agents and start building the UnderCover platform.<br/><br/>
            <b>I have a quick request for you. If you have a moment to rate the course, I'd hugely appreciate it. It's one of the main ways Udemy decides whether
            to show the course to others, and I'd be super grateful.</b>
            </span>
        </td>
    </tr>
</table>